# Tabela Silver — `ecommerce_enderecos`

Este notebook aplica regras de qualidade usando **PySpark**, separa registros válidos e rejeitados, e envia ambos para tabelas separadas no SQL Server.

A tabela Silver contém todos os registros, com colunas de auditoria:

- `invalidado`: `Sim` ou `Não`
- `is_valido`: `S` ou `N`
- `motivo_rejeicao`
- `data_hora_rejeicao`

Assim, registros inválidos não são descartados e também não é necessário criar uma tabela separada de rejeitados.

## Imports e parâmetros

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from functools import reduce
import uuid

RUN_ID = str(uuid.uuid4())

TABELA_SILVER = "squad1.silver_ecommerce_enderecos"
TABELA_DQ_LOGS = "squad1.dq_monitoring_logs"
TABELA_BRONZE_ENDERECOS = "squad1.bronze_ecommerce_enderecos"
TABELA_BRONZE_CLIENTES = "squad1.bronze_ecommerce_clientes"
TABELA_SILVER_CLIENTES = "squad1.silver_ecommerce_clientes"

NOME_TABELA_DQ = "silver_ecommerce_enderecos"

print("RUN_ID:", RUN_ID)
print("Tabela Silver:", TABELA_SILVER)
print("Tabela DQ Logs:", TABELA_DQ_LOGS)



## Funções auxiliares


In [0]:
def tabela_existe(nome_tabela: str) -> bool:
    try:
        return spark.catalog.tableExists(nome_tabela)
    except Exception:
        try:
            spark.table(nome_tabela).limit(1).count()
            return True
        except Exception:
            return False


def ler_tabela_delta(nome_tabela: str):
    if not tabela_existe(nome_tabela):
        raise Exception(f"Tabela Delta não encontrada: {nome_tabela}")
    return spark.table(nome_tabela)


def anti_duplicidade_por_arquivo(df_novo, tabela_destino: str, coluna_arquivo: str = "bronze_source_file"):
    if not tabela_existe(tabela_destino):
        print(f"Tabela {tabela_destino} ainda não existe. Todo micro-lote será processado.")
        return df_novo

    df_processados = (
        spark.table(tabela_destino)
        .select(coluna_arquivo)
        .where(F.col(coluna_arquivo).isNotNull())
        .dropDuplicates()
    )

    return df_novo.join(df_processados, on=coluna_arquivo, how="left_anti")


def salvar_delta_append(df, nome_tabela: str):
    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(nome_tabela)
    )
    print(f"Dados gravados em {nome_tabela}")


def criar_tabela_dq_logs_se_nao_existir():
    schema_dq_logs = StructType([
        StructField("run_id", StringType(), False),
        StructField("tabela", StringType(), False),
        StructField("regra", StringType(), False),
        StructField("status", StringType(), False),
        StructField("severidade", StringType(), False),
        StructField("qtd_registros_falhos", IntegerType(), False),
        StructField("qtd_registros_total", IntegerType(), False),
        StructField("timestamp_execucao", TimestampType(), False),
        StructField("arquivo_origem", StringType(), False),
    ])

    if not spark.catalog.tableExists(TABELA_DQ_LOGS):
        df_empty_logs = spark.createDataFrame([], schema_dq_logs)
        (
            df_empty_logs.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(TABELA_DQ_LOGS)
        )
        print(f"Tabela Delta criada: {TABELA_DQ_LOGS}")
    else:
        print(f"Tabela Delta já existe: {TABELA_DQ_LOGS}")


## Ler Bronze de endereços e selecionar apenas micro-lotes novos


In [0]:
df_bronze_enderecos = ler_tabela_delta(TABELA_BRONZE_ENDERECOS)

if "bronze_source_file" not in df_bronze_enderecos.columns:
    raise Exception("A Bronze precisa conter a coluna bronze_source_file.")

if "bronze_ingested_at" not in df_bronze_enderecos.columns:
    raise Exception("A Bronze precisa conter a coluna bronze_ingested_at.")

df_micro_lote = anti_duplicidade_por_arquivo(
    df_novo=df_bronze_enderecos,
    tabela_destino=TABELA_SILVER,
    coluna_arquivo="bronze_source_file"
)

qtd_micro_lote = df_micro_lote.count()
print("Registros novos para processar:", qtd_micro_lote)

if qtd_micro_lote == 0:
    dbutils.notebook.exit("Nenhum arquivo novo para processar na Silver de endereços.")

display(
    df_micro_lote
    .select("bronze_source_file")
    .dropDuplicates()
    .orderBy("bronze_source_file")
)


## Padronização mínima para validação


In [0]:
df_base = (
    df_micro_lote
    .withColumn("id_endereco_str", F.trim(F.col("id_endereco").cast("string")))
    .withColumn("id_cliente_str", F.trim(F.col("id_cliente").cast("string")))
    .withColumn("cep_digits", F.regexp_replace(F.col("cep").cast("string"), r"\D", ""))
    .withColumn("estado_norm", F.upper(F.trim(F.col("estado").cast("string"))))
    .withColumn("latitude_num", F.col("latitude").cast("double"))
    .withColumn("longitude_num", F.col("longitude").cast("double"))
    .withColumn("apelido_norm", F.initcap(F.lower(F.trim(F.col("apelido").cast("string")))))
    .withColumn("logradouro_norm", F.trim(F.col("logradouro").cast("string")))
    .withColumn("numero_str", F.trim(F.col("numero").cast("string")))
    .withColumn(
        "is_principal_bool",
        F.when(F.lower(F.trim(F.col("is_principal").cast("string"))).isin("true", "1", "sim", "s"), F.lit(True))
         .otherwise(F.lit(False))
    )
)


## 4. Resumo das rejeições por regra

In [0]:
df_resumo_rejeicoes = (
    df_validado
    .filter(F.size(F.col("motivos_array")) > 0)
    .select(F.explode("motivos_array").alias("regra"))
    .groupBy("regra")
    .count()
    .orderBy("regra")
)

display(df_resumo_rejeicoes)


## Preparar referências externas


In [0]:
# Regra 2: id_cliente deve existir em ecommerce_clientes.
# Preferimos a Silver de clientes, se existir. Caso contrário, usamos a Bronze de clientes como referência.
if tabela_existe(TABELA_SILVER_CLIENTES):
    tabela_ref_clientes = TABELA_SILVER_CLIENTES
elif tabela_existe(TABELA_BRONZE_CLIENTES):
    tabela_ref_clientes = TABELA_BRONZE_CLIENTES
else:
    tabela_ref_clientes = None

if tabela_ref_clientes:
    print("Referência de clientes usada:", tabela_ref_clientes)
    df_clientes_ref = (
        spark.table(tabela_ref_clientes)
        .select(F.trim(F.col("id_cliente").cast("string")).alias("id_cliente_str"))
        .where(F.col("id_cliente_str").isNotNull() & (F.col("id_cliente_str") != ""))
        .dropDuplicates()
        .withColumn("cliente_existe", F.lit(True))
    )
else:
    print("Aviso: nenhuma tabela de clientes encontrada. Regra 2 marcará todos como falha.")
    df_clientes_ref = spark.createDataFrame([], "id_cliente_str string, cliente_existe boolean")

# Para as regras 1, 6 e 8, usamos o histórico já existente na Silver, quando existir,
# para não validar apenas o micro-lote isoladamente.
if tabela_existe(TABELA_SILVER):
    df_silver_existente = spark.table(TABELA_SILVER)

    df_ids_endereco_existentes = (
        df_silver_existente
        .select(F.trim(F.col("id_endereco").cast("string")).alias("id_endereco_str"))
        .where(F.col("id_endereco_str").isNotNull() & (F.col("id_endereco_str") != ""))
        .dropDuplicates()
        .withColumn("id_endereco_ja_existe", F.lit(True))
    )

    df_cliente_resumo_existente = (
        df_silver_existente
        .withColumn("id_cliente_str", F.trim(F.col("id_cliente").cast("string")))
        .withColumn(
            "is_principal_bool_existente",
            F.when(F.lower(F.trim(F.col("is_principal").cast("string"))).isin("true", "1", "sim", "s"), F.lit(True))
             .otherwise(F.lit(False))
        )
        .where(F.col("id_cliente_str").isNotNull() & (F.col("id_cliente_str") != ""))
        .groupBy("id_cliente_str")
        .agg(
            F.count("*").alias("qtd_enderecos_existentes"),
            F.sum(F.when(F.col("is_principal_bool_existente"), 1).otherwise(0)).alias("qtd_principal_existente")
        )
    )
else:
    df_ids_endereco_existentes = spark.createDataFrame([], "id_endereco_str string, id_endereco_ja_existe boolean")
    df_cliente_resumo_existente = spark.createDataFrame([], "id_cliente_str string, qtd_enderecos_existentes long, qtd_principal_existente long")


## Aplicar as 10 regras como flags booleanas


In [0]:
w_id_endereco = Window.partitionBy("id_endereco_str")
w_cliente = Window.partitionBy("id_cliente_str")

ufs_validas = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG",
    "PA", "PB", "PR", "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"
]

apelidos_validos = ["Casa", "Trabalho", "Outro"]

# Contagens internas do micro-lote
# Regras 1, 6 e 8 dependem dessas contagens.
df_contagens_lote = (
    df_base
    .withColumn("qtd_id_endereco_no_lote", F.count("*").over(w_id_endereco))
    .withColumn("qtd_enderecos_cliente_no_lote", F.count("*").over(w_cliente))
    .withColumn(
        "qtd_principal_cliente_no_lote",
        F.sum(F.when(F.col("is_principal_bool"), 1).otherwise(0)).over(w_cliente)
    )
)

df_silver_enderecos = (
    df_contagens_lote
    .join(df_clientes_ref, on="id_cliente_str", how="left")
    .join(df_ids_endereco_existentes, on="id_endereco_str", how="left")
    .join(df_cliente_resumo_existente, on="id_cliente_str", how="left")
    .withColumn("cliente_existe", F.coalesce(F.col("cliente_existe"), F.lit(False)))
    .withColumn("id_endereco_ja_existe", F.coalesce(F.col("id_endereco_ja_existe"), F.lit(False)))
    .withColumn("qtd_enderecos_existentes", F.coalesce(F.col("qtd_enderecos_existentes"), F.lit(0)))
    .withColumn("qtd_principal_existente", F.coalesce(F.col("qtd_principal_existente"), F.lit(0)))
    .withColumn("qtd_enderecos_cliente_total", F.col("qtd_enderecos_existentes") + F.col("qtd_enderecos_cliente_no_lote"))
    .withColumn("qtd_principal_cliente_total", F.col("qtd_principal_existente") + F.col("qtd_principal_cliente_no_lote"))

    # R1: id_endereco não pode ser nulo nem duplicado
    .withColumn(
        "r1_id_endereco_falhou",
        F.col("id_endereco_str").isNull()
        | (F.col("id_endereco_str") == "")
        | (F.col("qtd_id_endereco_no_lote") > 1)
        | F.col("id_endereco_ja_existe")
    )

    # R2: id_cliente deve existir em ecommerce_clientes
    .withColumn(
        "r2_id_cliente_fk_falhou",
        F.col("id_cliente_str").isNull()
        | (F.col("id_cliente_str") == "")
        | (~F.col("cliente_existe"))
    )

    # R3: cep deve ter exatamente 8 dígitos numéricos
    .withColumn(
        "r3_cep_falhou",
        F.col("cep_digits").isNull()
        | (~F.col("cep_digits").rlike(r"^\d{8}$"))
    )

    # R4: estado deve ser UF válida
    .withColumn(
        "r4_estado_falhou",
        F.col("estado_norm").isNull()
        | (~F.col("estado_norm").rlike(r"^[A-Z]{2}$"))
        | (~F.col("estado_norm").isin(ufs_validas))
    )

    # R5: latitude e longitude não podem ser nulas e devem estar no range do Brasil
    .withColumn(
        "r5_lat_lon_falhou",
        F.col("latitude_num").isNull()
        | F.col("longitude_num").isNull()
        | (~F.col("latitude_num").between(-33.75, 5.27))
        | (~F.col("longitude_num").between(-73.99, -32.39))
    )

    # R6: cada cliente deve ter exatamente 1 endereço principal
    .withColumn(
        "r6_endereco_principal_falhou",
        F.col("id_cliente_str").isNull()
        | (F.col("id_cliente_str") == "")
        | (F.col("qtd_principal_cliente_total") != 1)
    )

    # R7: apelido deve ser Casa, Trabalho ou Outro
    .withColumn(
        "r7_apelido_falhou",
        F.col("apelido_norm").isNull()
        | (~F.col("apelido_norm").isin(apelidos_validos))
    )

    # R8: nenhum cliente deve ter mais de 3 endereços cadastrados
    .withColumn(
        "r8_max_3_enderecos_falhou",
        F.col("id_cliente_str").isNull()
        | (F.col("id_cliente_str") == "")
        | (F.col("qtd_enderecos_cliente_total") > 3)
    )

    # R9: logradouro não pode ser nulo ou vazio
    .withColumn(
        "r9_logradouro_falhou",
        F.col("logradouro_norm").isNull()
        | (F.col("logradouro_norm") == "")
    )

    # R10: numero não pode ser nulo
    .withColumn(
        "r10_numero_falhou",
        F.col("numero").isNull()
        | (F.col("numero_str") == "")
    )

    .withColumn("silver_processed_at", F.current_timestamp())
    .withColumn(
        "silver_linha_valida",
        ~(
            F.col("r1_id_endereco_falhou")
            | F.col("r2_id_cliente_fk_falhou")
            | F.col("r3_cep_falhou")
            | F.col("r4_estado_falhou")
            | F.col("r5_lat_lon_falhou")
            | F.col("r6_endereco_principal_falhou")
            | F.col("r7_apelido_falhou")
            | F.col("r8_max_3_enderecos_falhou")
            | F.col("r9_logradouro_falhou")
            | F.col("r10_numero_falhou")
        )
    )
)

# Remove colunas auxiliares de cálculo para deixar a Silver mais limpa.
colunas_auxiliares = [
    "id_endereco_str",
    "id_cliente_str",
    "cep_digits",
    "estado_norm",
    "latitude_num",
    "longitude_num",
    "apelido_norm",
    "logradouro_norm",
    "numero_str",
    "is_principal_bool",
    "qtd_id_endereco_no_lote",
    "qtd_enderecos_cliente_no_lote",
    "qtd_principal_cliente_no_lote",
    "cliente_existe",
    "id_endereco_ja_existe",
    "qtd_enderecos_existentes",
    "qtd_principal_existente",
    "qtd_enderecos_cliente_total",
    "qtd_principal_cliente_total",
]

for coluna in colunas_auxiliares:
    if coluna in df_silver_enderecos.columns:
        df_silver_enderecos = df_silver_enderecos.drop(coluna)

display(df_silver_enderecos.limit(10))


## Resumo da validação


In [0]:
flags_regras = [
    "r1_id_endereco_falhou",
    "r2_id_cliente_fk_falhou",
    "r3_cep_falhou",
    "r4_estado_falhou",
    "r5_lat_lon_falhou",
    "r6_endereco_principal_falhou",
    "r7_apelido_falhou",
    "r8_max_3_enderecos_falhou",
    "r9_logradouro_falhou",
    "r10_numero_falhou",
]

exprs = [F.sum(F.when(F.col(c), 1).otherwise(0)).alias(c) for c in flags_regras]

display(df_silver_enderecos.agg(*exprs))
display(df_silver_enderecos.groupBy("silver_linha_valida").count())


## Gerar `dq_monitoring_logs`


In [0]:
regras_config = [
    ("R1 - id_endereco não pode ser nulo nem duplicado", "r1_id_endereco_falhou", "Critica"),
    ("R2 - id_cliente deve existir em ecommerce_clientes", "r2_id_cliente_fk_falhou", "Critica"),
    ("R3 - cep deve ter exatamente 8 dígitos numéricos", "r3_cep_falhou", "Critica"),
    ("R4 - estado deve ser uma UF válida", "r4_estado_falhou", "Critica"),
    ("R5 - latitude e longitude devem estar no range do Brasil", "r5_lat_lon_falhou", "Critica"),
    ("R6 - cliente deve ter exatamente 1 endereço principal", "r6_endereco_principal_falhou", "Critica"),
    ("R7 - apelido deve ser Casa, Trabalho ou Outro", "r7_apelido_falhou", "Aviso"),
    ("R8 - cliente não deve ter mais de 3 endereços", "r8_max_3_enderecos_falhou", "Critica"),
    ("R9 - logradouro não pode ser nulo ou vazio", "r9_logradouro_falhou", "Critica"),
    ("R10 - numero não pode ser nulo", "r10_numero_falhou", "Critica"),
]


def criar_log_regra(df, nome_regra, coluna_flag, severidade):
    return (
        df
        .groupBy("bronze_source_file")
        .agg(
            F.count("*").cast("int").alias("qtd_registros_total"),
            F.sum(F.when(F.col(coluna_flag), 1).otherwise(0)).cast("int").alias("qtd_registros_falhos")
        )
        .withColumn("run_id", F.lit(RUN_ID))
        .withColumn("tabela", F.lit(NOME_TABELA_DQ))
        .withColumn("regra", F.lit(nome_regra))
        .withColumn("status", F.when(F.col("qtd_registros_falhos") > 0, F.lit("FAIL")).otherwise(F.lit("PASS")))
        .withColumn("severidade", F.lit(severidade))
        .withColumn("timestamp_execucao", F.current_timestamp())
        .withColumnRenamed("bronze_source_file", "arquivo_origem")
        .select(
            "run_id",
            "tabela",
            "regra",
            "status",
            "severidade",
            "qtd_registros_falhos",
            "qtd_registros_total",
            "timestamp_execucao",
            "arquivo_origem"
        )
    )

logs = [criar_log_regra(df_silver_enderecos, regra, flag, severidade) for regra, flag, severidade in regras_config]
df_dq_monitoring_logs = reduce(lambda a, b: a.unionByName(b), logs)

display(df_dq_monitoring_logs.orderBy("arquivo_origem", "regra"))


## Criar `dq_monitoring_logs`, se necessário, e evitar duplicidade de logs


In [0]:
criar_tabela_dq_logs_se_nao_existir()

# Como a tabela é compartilhada por várias squads, evitamos duplicidade pela chave:
# tabela + regra + arquivo_origem.
if tabela_existe(TABELA_DQ_LOGS):
    df_logs_existentes = (
        spark.table(TABELA_DQ_LOGS)
        .select("tabela", "regra", "arquivo_origem")
        .dropDuplicates()
    )

    df_dq_monitoring_logs_novos = (
        df_dq_monitoring_logs
        .join(
            df_logs_existentes,
            on=["tabela", "regra", "arquivo_origem"],
            how="left_anti"
        )
    )
else:
    df_dq_monitoring_logs_novos = df_dq_monitoring_logs

print("Logs DQ novos:", df_dq_monitoring_logs_novos.count())


## Gravar Silver e Logs DQ em Delta


In [0]:
salvar_delta_append(df_silver_enderecos, TABELA_SILVER)

if df_dq_monitoring_logs_novos.count() > 0:
    salvar_delta_append(df_dq_monitoring_logs_novos, TABELA_DQ_LOGS)
else:
    print("Nenhum log DQ novo para gravar.")


## Validação final


In [0]:
print("=" * 80)
print("SILVER ENDEREÇOS CONCLUÍDA")
print("RUN_ID:", RUN_ID)
print("Registros processados na Silver:", df_silver_enderecos.count())
print("Logs DQ gerados no run:", df_dq_monitoring_logs.count())
print("Logs DQ novos gravados:", df_dq_monitoring_logs_novos.count())
print("Tabela Silver:", TABELA_SILVER)
print("Tabela Logs:", TABELA_DQ_LOGS)
print("=" * 80)

print("Arquivos processados neste run:")
display(df_silver_enderecos.select("bronze_source_file").dropDuplicates().orderBy("bronze_source_file"))

print("Resumo DQ do run:")
display(
    df_dq_monitoring_logs
    .groupBy("status", "severidade")
    .agg(F.sum("qtd_registros_falhos").alias("falhas"), F.count("*").alias("logs"))
)
